In [1]:
import json
import pandas as pd
import requests
import time
import re
import multiprocessing
import math
from transformers import BertModel, BertConfig, BertTokenizer
import torch,gc
from collections import Counter
import os

### Common Function

In [2]:
def bert_define(path="/remote-home/cs_acmis_wsf/ai4dingo/model/bert_english_pretrained"):
    # 加载bert的tokenizer分词
    tokenizer = BertTokenizer.from_pretrained(path)
    # 加载预训练模型
    model_config = BertConfig.from_pretrained(path)
    model = BertModel.from_pretrained(path, config=model_config)
    model = model.cuda()
    return tokenizer, model

In [3]:
def closest_batch_size(vector_count, min_n=6, max_n=12):
    best_n = min_n
    best_batch_size = 2**best_n
    best_batch = vector_count // best_batch_size
    min_difference = abs(vector_count - best_batch * best_batch_size)

    for n in range(min_n + 1, max_n + 1):
        batch_size = 2**n
        batch = vector_count // batch_size
        difference = abs(vector_count - batch * batch_size)

        # 检查当前batch_size是否比之前的更接近vector_count
        if difference < min_difference:
            best_n = n
            best_batch_size = batch_size
            best_batch = batch
            min_difference = difference

    return best_n, best_batch_size, best_batch

In [4]:
def split_dataframe(df, n_parts, rows_per_part):
    # 计算总行数
    total_rows = len(df)
    # 计算前n-1份的总行数
    rows_for_first_n_minus_one_parts = (n_parts - 1) * rows_per_part
    # 确定是否需要额外的部分来存储剩余的数据
    extra_part_needed = total_rows > rows_for_first_n_minus_one_parts

    # 创建一个包含所有分片的列表
    dfs = []

    # 分配前n-1份
    for i in range(n_parts - 1):
        start = i * rows_per_part
        end = start + rows_per_part
        dfs.append(df.iloc[start:end])

    # 如果需要，分配最后一部分
    if extra_part_needed:
        start = (n_parts - 1) * rows_per_part
        dfs.append(df.iloc[start:])

    return dfs


In [5]:
def handle_text(text):
    text = text.replace('\n','').replace('\t','')
    return text
    

### handle user

In [7]:
user_data = []

# 打开JSON文件
with open('/remote-home/cs_acmis_wsf/ai4dingo/yelp/yelp_academic_dataset_user.json', 'r', encoding='utf-8') as file:
    # 逐行读取
    for line in file:
        # 解析每一行为JSON对象
        user = json.loads(line)
        user_id = user["user_id"]
        name = user["name"]
        review_count = user["review_count"]
        yelping_since = user["yelping_since"]
        useful = user["useful"]
        funny = user["funny"]
        cool = user["cool"]
        elite = user["elite"]
        fans = user["fans"]
        average_stars = user["average_stars"]
        compliment_hot = user["compliment_hot"]
        compliment_more = user["compliment_more"]
        compliment_profile = user["compliment_profile"]
        compliment_cute = user["compliment_cute"]
        compliment_list = user["compliment_list"]
        compliment_note = user["compliment_note"]
        compliment_plain = user["compliment_plain"]
        compliment_cool = user["compliment_cool"]
        compliment_funny = user["compliment_funny"]
        compliment_writer = user["compliment_writer"]
        compliment_photos = user["compliment_photos"]
        user_data.append([user_id,name,review_count,yelping_since,useful,funny,cool,elite,fans,average_stars,compliment_hot,compliment_more,compliment_profile
                         ,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos])


In [8]:
len(user_data)

1987897

In [9]:
user_df = pd.DataFrame(user_data,columns=["user_id","name","review_count","yelping_since","useful","funny","cool","elite","fans","average_stars","compliment_hot","compliment_more","compliment_profile"
                         ,"compliment_cute","compliment_list","compliment_note","compliment_plain","compliment_cool","compliment_funny","compliment_writer","compliment_photos"])

### handle business

In [16]:
business_data = []

# 打开JSON文件
with open('/remote-home/cs_acmis_wsf/ai4dingo/yelp/yelp_academic_dataset_business.json', 'r', encoding='utf-8') as file:
    # 逐行读取
    for line in file:
        # 解析每一行为JSON对象
        business = json.loads(line)
        business_id = business["business_id"]
        name = business["name"]
        address = business["address"]
        city = business["city"]
        state = business["state"]
        postal_code = business["postal_code"]
        stars = business["stars"]
        review_count = business["review_count"]
        categories = business["categories"]
        business_data.append([business_id,name,address,city,state,postal_code,stars,review_count,categories])

In [64]:
business_df = pd.DataFrame(business_data,columns=["business_id","name","address","city","state","postal_code","stars","review_count","categories"])

In [65]:
business_df['length'] = business_df['categories'].str.len()

In [67]:
business_df = business_df.dropna(subset=['categories'])

In [68]:
business_df

,business_id,name,address,city,state,postal_code,stars,review_count,categories,length
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,5.0,7,"Doctors, Traditional Chinese Medicine, Naturop...",106.0
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,87 Grasso Plaza Shopping Center,Affton,MO,63123,3.0,15,"Shipping Centers, Local Services, Notaries, Ma...",78.0
2,tUFrWirKiKi_TAnsVWINQQ,Target,5255 E Broadway Blvd,Tucson,AZ,85711,3.5,22,"Department Stores, Shopping, Fashion, Home & G...",82.0
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,4.0,80,"Restaurants, Food, Bubble Tea, Coffee & Tea, B...",53.0
4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,101 Walnut St,Green Lane,PA,18054,4.5,13,"Brewpubs, Breweries, Food",25.0
...,...,...,...,...,...,...,...,...,...,...
150341,IUQopTMmYQG-qRtBk-8QnA,Binh's Nails,3388 Gateway Blvd,Edmonton,AB,T6J 5H2,3.0,13,"Nail Salons, Beauty & Spas",26.0
150342,c8GjPIOTGVmIemT7j5_SyQ,Wild Birds Unlimited,2813 Bransford Ave,Nashville,TN,37204,4.0,5,"Pets, Nurseries & Gardening, Pet Stores, Hobby...",89.0
150343,_QAMST-NrQobXduilWEqSw,Claire's Boutique,"6020 E 82nd St, Ste 46",Indianapolis,IN,46250,3.5,8,"Shopping, Jewelry, Piercing, Toy Stores, Beaut...",76.0
150344,mtGm22y5c2UHNXDFAjaPNw,Cyclery & Fitness Center,2472 Troy Rd,Edwardsville,IL,62025,4.0,24,"Fitness/Exercise Equipment, Eyewear & Optician...",80.0


In [32]:
tokenizer,model = bert_define()

Some weights of the model checkpoint at /remote-home/cs_acmis_wsf/ai4dingo/model/bert_english_pretrained were not used when initializing BertModel: ['cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.bias', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight', 'cls.predictions.decoder.weight', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.dense.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [33]:
# 构成一个小batch
batch_token, batch_segment, batch_mask = list(), list(), list()

In [74]:
text = business_df["categories"].tolist()

In [75]:
max_len = int(business_df["length"].max())

In [76]:
for t in text:
    # text 作 tokenizer 分词
    token = tokenizer.tokenize(t)
    token = ['[CLS]'] + token + ['[SEP]']
    token_id = tokenizer.convert_tokens_to_ids(token)  # 字转换vocab中的index 
    # 加padding补齐及segment、mask
    padding = [0] * (max_len - len(token_id))
    mask = [1] * len(token_id) + padding
    segment = [0] * len(token_id) + padding
    token_id = token_id + padding

    batch_token.append(token_id)
    batch_segment.append(segment)
    batch_mask.append(mask)


In [79]:
vector_count = min([len(business_df),len(batch_mask),len(batch_segment),len(batch_token)])

In [80]:
vector_count

150243

In [82]:
n, batch_size, batch = closest_batch_size(vector_count)
print(f"vector_count: {vector_count}, n: {n}, batch_size: {batch_size}, batch: {batch}")

vector_count: 150243, n: 6, batch_size: 64, batch: 2347


In [97]:
rows_per_part = 5120
n =  math.ceil(vector_count / rows_per_part)

In [98]:
split_dfs_business = split_dataframe(business_df, n,rows_per_part)

##### 循环处理

In [100]:
batch_size_num = 64

for i in range(len(split_dfs_business)):
    df = split_dfs_business[i]
    print(f'正在读取第 {i+1} 个DF')
    text = df["categories"].tolist()
    
    print(f'初始化batch数据： {i+1}')
    batch_token, batch_segment, batch_mask = list(), list(), list()
    for t in text:
        # text 作 tokenizer 分词
        token = tokenizer.tokenize(t)
        token = ['[CLS]'] + token + ['[SEP]']
        token_id = tokenizer.convert_tokens_to_ids(token)  # 字转换vocab中的index

        # 加padding补齐及segment、mask
        padding = [0] * (max_len - len(token_id))
        mask = [1] * len(token_id) + padding
        segment = [0] * len(token_id) + padding
        token_id = token_id + padding

        batch_token.append(token_id)
        batch_segment.append(segment)
        batch_mask.append(mask)
        
    print(f'batch数据 组装完毕 ：{i+1}')
   
    batch_tensor_token = torch.tensor(batch_token)
    batch_tensor_segment = torch.tensor(batch_segment)
    batch_tensor_mask = torch.tensor(batch_mask)

    batch_len = len(batch_tensor_token)
    
    if torch.cuda.is_available():
        batch_tensor_token = batch_tensor_token.to('cuda:0')
        batch_tensor_segment = batch_tensor_segment.to('cuda:0')
        batch_tensor_mask = batch_tensor_mask.to('cuda:0')
    
    print(f'batch_tensor数据 组装完毕 ：{i+1},开始分块处理')
    
    batch_tensor_token_chunked = [batch_tensor_token[i:i+batch_size_num] for i in range(0,batch_len,batch_size_num)]

    batch_tensor_segment_chunked = [batch_tensor_segment[i:i+batch_size_num] for i in range(0,batch_len,batch_size_num)]

    batch_tensor_mask_chunked = [batch_tensor_mask[i:i+batch_size_num] for i in range(0,batch_len,batch_size_num)]

    print(f'分块处理完毕数据完毕，每块的长度为:{batch_size_num},一共有 {len(batch_tensor_token_chunked)} 块， 开始调用模型')
    
    gc.collect()
    torch.cuda.empty_cache()
    text_vector = []
    
    try:
        for k in range(len(batch_tensor_token_chunked)):
            with torch.no_grad():
                print(f'正在处理 {i+1} ----- {k+1}')
                outputs = model(batch_tensor_token_chunked[k], token_type_ids=batch_tensor_segment_chunked[k], attention_mask=batch_tensor_mask_chunked[k])
                outputs = outputs[0][:, 0, :]  # 取cls向量
                for o in range(outputs.shape[0]):
                    text_vector.append(outputs[o])
        df["feature"] = [ tv.tolist() for tv in  text_vector] 
        df.to_csv(f'/remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/business_{i+1}.csv', index=False)
        print(f"文件写入完毕 {i+1}")
                    
    except Exception as e:
        print(f"出现异常 {i+1}: {e}")
        continue
            
    
    print('============================')
    print()

正在读取第 1 个DF
初始化batch数据： 1
batch数据 组装完毕 ：1
batch_tensor数据 组装完毕 ：1,开始分块处理
分块处理完毕数据完毕，每块的长度为:64,一共有 80 块， 开始调用模型
正在处理 1 ----- 1
正在处理 1 ----- 2
正在处理 1 ----- 3
正在处理 1 ----- 4
正在处理 1 ----- 5
正在处理 1 ----- 6
正在处理 1 ----- 7
正在处理 1 ----- 8
正在处理 1 ----- 9
正在处理 1 ----- 10
正在处理 1 ----- 11
正在处理 1 ----- 12
正在处理 1 ----- 13
正在处理 1 ----- 14
正在处理 1 ----- 15
正在处理 1 ----- 16
正在处理 1 ----- 17
正在处理 1 ----- 18
正在处理 1 ----- 19
正在处理 1 ----- 20
正在处理 1 ----- 21
正在处理 1 ----- 22
正在处理 1 ----- 23
正在处理 1 ----- 24
正在处理 1 ----- 25
正在处理 1 ----- 26
正在处理 1 ----- 27
正在处理 1 ----- 28
正在处理 1 ----- 29
正在处理 1 ----- 30
正在处理 1 ----- 31
正在处理 1 ----- 32
正在处理 1 ----- 33
正在处理 1 ----- 34
正在处理 1 ----- 35
正在处理 1 ----- 36
正在处理 1 ----- 37
正在处理 1 ----- 38
正在处理 1 ----- 39
正在处理 1 ----- 40
正在处理 1 ----- 41
正在处理 1 ----- 42
正在处理 1 ----- 43
正在处理 1 ----- 44
正在处理 1 ----- 45
正在处理 1 ----- 46
正在处理 1 ----- 47
正在处理 1 ----- 48
正在处理 1 ----- 49
正在处理 1 ----- 50
正在处理 1 ----- 51
正在处理 1 ----- 52
正在处理 1 ----- 53
正在处理 1 ----- 54
正在处理 1 ----- 55
正在处理 1 ----- 56
正在处

/tmp/ipykernel_515/376242564.py:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["feature"] = [ tv.tolist() for tv in  text_vector]


文件写入完毕 1

正在读取第 2 个DF
初始化batch数据： 2
batch数据 组装完毕 ：2
batch_tensor数据 组装完毕 ：2,开始分块处理
分块处理完毕数据完毕，每块的长度为:64,一共有 80 块， 开始调用模型
正在处理 2 ----- 1
正在处理 2 ----- 2
正在处理 2 ----- 3
正在处理 2 ----- 4
正在处理 2 ----- 5
正在处理 2 ----- 6
正在处理 2 ----- 7
正在处理 2 ----- 8
正在处理 2 ----- 9
正在处理 2 ----- 10
正在处理 2 ----- 11
正在处理 2 ----- 12
正在处理 2 ----- 13
正在处理 2 ----- 14
正在处理 2 ----- 15
正在处理 2 ----- 16
正在处理 2 ----- 17
正在处理 2 ----- 18
正在处理 2 ----- 19
正在处理 2 ----- 20
正在处理 2 ----- 21
正在处理 2 ----- 22
正在处理 2 ----- 23
正在处理 2 ----- 24
正在处理 2 ----- 25
正在处理 2 ----- 26
正在处理 2 ----- 27
正在处理 2 ----- 28
正在处理 2 ----- 29
正在处理 2 ----- 30
正在处理 2 ----- 31
正在处理 2 ----- 32
正在处理 2 ----- 33
正在处理 2 ----- 34
正在处理 2 ----- 35
正在处理 2 ----- 36
正在处理 2 ----- 37
正在处理 2 ----- 38
正在处理 2 ----- 39
正在处理 2 ----- 40
正在处理 2 ----- 41
正在处理 2 ----- 42
正在处理 2 ----- 43
正在处理 2 ----- 44
正在处理 2 ----- 45
正在处理 2 ----- 46
正在处理 2 ----- 47
正在处理 2 ----- 48
正在处理 2 ----- 49
正在处理 2 ----- 50
正在处理 2 ----- 51
正在处理 2 ----- 52
正在处理 2 ----- 53
正在处理 2 ----- 54
正在处理 2 ----- 55
正在处理 2 --

/tmp/ipykernel_515/376242564.py:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["feature"] = [ tv.tolist() for tv in  text_vector]


文件写入完毕 2

正在读取第 3 个DF
初始化batch数据： 3
batch数据 组装完毕 ：3
batch_tensor数据 组装完毕 ：3,开始分块处理
分块处理完毕数据完毕，每块的长度为:64,一共有 80 块， 开始调用模型
正在处理 3 ----- 1
正在处理 3 ----- 2
正在处理 3 ----- 3
正在处理 3 ----- 4
正在处理 3 ----- 5
正在处理 3 ----- 6
正在处理 3 ----- 7
正在处理 3 ----- 8
正在处理 3 ----- 9
正在处理 3 ----- 10
正在处理 3 ----- 11
正在处理 3 ----- 12
正在处理 3 ----- 13
正在处理 3 ----- 14
正在处理 3 ----- 15
正在处理 3 ----- 16
正在处理 3 ----- 17
正在处理 3 ----- 18
正在处理 3 ----- 19
正在处理 3 ----- 20
正在处理 3 ----- 21
正在处理 3 ----- 22
正在处理 3 ----- 23
正在处理 3 ----- 24
正在处理 3 ----- 25
正在处理 3 ----- 26
正在处理 3 ----- 27
正在处理 3 ----- 28
正在处理 3 ----- 29
正在处理 3 ----- 30
正在处理 3 ----- 31
正在处理 3 ----- 32
正在处理 3 ----- 33
正在处理 3 ----- 34
正在处理 3 ----- 35
正在处理 3 ----- 36
正在处理 3 ----- 37
正在处理 3 ----- 38
正在处理 3 ----- 39
正在处理 3 ----- 40
正在处理 3 ----- 41
正在处理 3 ----- 42
正在处理 3 ----- 43
正在处理 3 ----- 44
正在处理 3 ----- 45
正在处理 3 ----- 46
正在处理 3 ----- 47
正在处理 3 ----- 48
正在处理 3 ----- 49
正在处理 3 ----- 50
正在处理 3 ----- 51
正在处理 3 ----- 52
正在处理 3 ----- 53
正在处理 3 ----- 54
正在处理 3 ----- 55
正在处理 3 --

...

文件写入完毕 28

正在读取第 29 个DF
初始化batch数据： 29
batch数据 组装完毕 ：29
batch_tensor数据 组装完毕 ：29,开始分块处理
分块处理完毕数据完毕，每块的长度为:64,一共有 80 块， 开始调用模型
正在处理 29 ----- 1
正在处理 29 ----- 2
正在处理 29 ----- 3
正在处理 29 ----- 4
正在处理 29 ----- 5
正在处理 29 ----- 6
正在处理 29 ----- 7
正在处理 29 ----- 8
正在处理 29 ----- 9
正在处理 29 ----- 10
正在处理 29 ----- 11
正在处理 29 ----- 12
正在处理 29 ----- 13
正在处理 29 ----- 14
正在处理 29 ----- 15
正在处理 29 ----- 16
正在处理 29 ----- 17
正在处理 29 ----- 18
正在处理 29 ----- 19
正在处理 29 ----- 20
正在处理 29 ----- 21
正在处理 29 ----- 22
正在处理 29 ----- 23
正在处理 29 ----- 24
正在处理 29 ----- 25
正在处理 29 ----- 26
正在处理 29 ----- 27
正在处理 29 ----- 28
正在处理 29 ----- 29
正在处理 29 ----- 30
正在处理 29 ----- 31
正在处理 29 ----- 32
正在处理 29 ----- 33
正在处理 29 ----- 34
正在处理 29 ----- 35
正在处理 29 ----- 36
正在处理 29 ----- 37
正在处理 29 ----- 38
正在处理 29 ----- 39
正在处理 29 ----- 40
正在处理 29 ----- 41
正在处理 29 ----- 42
正在处理 29 ----- 43
正在处理 29 ----- 44
正在处理 29 ----- 45
正在处理 29 ----- 46
正在处理 29 ----- 47
正在处理 29 ----- 48
正在处理 29 ----- 49
正在处理 29 ----- 50
正在处理 29 ----- 51
正在处理 29 ----- 52


/tmp/ipykernel_515/376242564.py:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["feature"] = [ tv.tolist() for tv in  text_vector]


文件写入完毕 29

正在读取第 30 个DF
初始化batch数据： 30
batch数据 组装完毕 ：30
batch_tensor数据 组装完毕 ：30,开始分块处理
分块处理完毕数据完毕，每块的长度为:64,一共有 28 块， 开始调用模型
正在处理 30 ----- 1
正在处理 30 ----- 2
正在处理 30 ----- 3
正在处理 30 ----- 4
正在处理 30 ----- 5
正在处理 30 ----- 6
正在处理 30 ----- 7
正在处理 30 ----- 8
正在处理 30 ----- 9
正在处理 30 ----- 10
正在处理 30 ----- 11
正在处理 30 ----- 12
正在处理 30 ----- 13
正在处理 30 ----- 14
正在处理 30 ----- 15
正在处理 30 ----- 16
正在处理 30 ----- 17
正在处理 30 ----- 18
正在处理 30 ----- 19
正在处理 30 ----- 20
正在处理 30 ----- 21
正在处理 30 ----- 22
正在处理 30 ----- 23
正在处理 30 ----- 24
正在处理 30 ----- 25
正在处理 30 ----- 26
正在处理 30 ----- 27
正在处理 30 ----- 28


/tmp/ipykernel_515/376242564.py:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["feature"] = [ tv.tolist() for tv in  text_vector]


文件写入完毕 30



#### 文件合并

In [100]:
n

208

In [106]:
folder_path = '/remote-home/cs_acmis_wsf/ai4dingo/yelp/vector'
file_names = [f'review_{i}.csv' for i in range(1, 1079)]

In [103]:
# 读取并合并所有CSV文件
df_combined = pd.concat((pd.read_csv(os.path.join(folder_path, file_name)) for file_name in file_names), ignore_index=True)

# 将合并后的DataFrame保存为新的CSV文件
output_file = '/remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/tip.csv'
df_combined.to_csv(output_file, index=False)

In [75]:
### 每100w存一次

In [76]:
# 初始化变量
chunksize = 1000000
df_combined = pd.DataFrame()
output_file_base = '/remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_part'
file_index = 1
row_count = 0

# 读取并合并所有CSV文件
for file_name in file_names:
    file_path = os.path.join(folder_path, file_name)
    for chunk in pd.read_csv(file_path, chunksize=chunksize):
        df_combined = pd.concat([df_combined, chunk], ignore_index=True)
        row_count += len(chunk)

        if row_count >= chunksize:
            output_file = f'{output_file_base}_{file_index}.csv'
            df_combined.to_csv(output_file, index=False)
            df_combined = pd.DataFrame()
            row_count = 0
            file_index += 1

# 保存剩余的DataFrame
if not df_combined.empty:
    output_file = f'{output_file_base}_{file_index}.csv'
    df_combined.to_csv(output_file, index=False)
    

In [107]:
# 删除原始的30个文件
for file_name in file_names:
    file_path = os.path.join(folder_path, file_name)
    try:
        os.remove(file_path)
        print(f"文件 {file_path} 已删除。")
    except FileNotFoundError:
        print(f"文件 {file_path} 不存在，无法删除。")

文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_1.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_2.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_3.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_4.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_5.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_6.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_7.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_8.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_9.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_10.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_11.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_12.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_13.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_14.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/

文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_192.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_193.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_194.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_195.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_196.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_197.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_198.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_199.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_200.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_201.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_202.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_203.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_204.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_205.csv 已删除。
文件 /remote-home/cs_a

文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_435.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_436.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_437.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_438.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_439.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_440.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_441.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_442.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_443.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_444.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_445.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_446.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_447.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_448.csv 已删除。
文件 /remote-home/cs_a

文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_611.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_612.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_613.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_614.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_615.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_616.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_617.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_618.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_619.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_620.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_621.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_622.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_623.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_624.csv 已删除。
文件 /remote-home/cs_a

文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_788.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_789.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_790.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_791.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_792.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_793.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_794.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_795.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_796.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_797.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_798.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_799.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_800.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_801.csv 已删除。
文件 /remote-home/cs_a

文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_914.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_915.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_916.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_917.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_918.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_919.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_920.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_921.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_922.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_923.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_924.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_925.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_926.csv 已删除。
文件 /remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_927.csv 已删除。
文件 /remote-home/cs_a

In [109]:
demo = df = pd.read_csv('/remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/business.csv')

In [115]:
feature = demo.iloc[4]["feature"]

In [120]:
s = "select name,stars,id_1,feature_index_1$distance from vector(business,feature,array"+feature+",10, map[efSearch, 768]) order by feature_index_1$distance limit 10;"

In [121]:
s

'select name,stars,id_1,feature_index_1$distance from vector(business,feature,array[0.6412854790687561, 0.15557944774627686, -0.25583526492118835, -0.37156498432159424, -0.21899624168872833, -0.13379773497581482, 0.3629712462425232, 0.3158617913722992, 0.07918655127286911, -1.2644082307815552, -0.4443523585796356, 0.2287846803665161, -0.199891597032547, -0.2042980194091797, -0.4010559916496277, 0.1766739785671234, -0.04298792779445648, 0.2203962504863739, -0.2398119419813156, -0.27657991647720337, 0.059902504086494446, -0.029948631301522255, 0.7761920690536499, -0.06299848109483719, 0.06768635660409927, 0.06226615235209465, -0.012628177180886269, 0.2853887975215912, -0.05010503530502319, 0.020810255780816078, -0.1265525370836258, 0.43903887271881104, -0.06854730099439621, 0.13895283639431, -0.029223991557955742, 0.05612121894955635, -0.17995919287204742, -0.5310277342796326, -0.15090595185756683, -0.06863129138946533, -0.679302990436554, 0.06805437803268433, 0.34885767102241516, 0.0494

### handle review

In [37]:
review_data = []

# 打开JSON文件
with open('/remote-home/cs_acmis_wsf/ai4dingo/yelp/yelp_academic_dataset_review.json', 'r', encoding='utf-8') as file:
    # 逐行读取
    for line in file:
        # 解析每一行为JSON对象
        review = json.loads(line)
        review_id = review["review_id"]
        user_id = review["user_id"]
        business_id = review["business_id"]
        stars = review["stars"]
        useful = review["useful"]
        funny = review["funny"]
        cool = review["cool"]
        text = review["text"]
        review_date = review["date"]
        r = [review_id,user_id,business_id,stars,useful,funny,cool,text,review_date]
        review_data.append(r)

In [38]:
len(review_data)

6990280

In [39]:
review_df = pd.DataFrame(review_data,columns=["review_id","user_id","business_id","stars","useful","funny","cool","text","review_data"])

In [40]:
review_df= review_df.drop_duplicates(subset='text', keep='last')

In [41]:
review_df["text"] = review_df["text"].apply(handle_text)

In [42]:
review_df = review_df.dropna(subset=['text'])

In [43]:
review_df['length'] = review_df['text'].str.len()

In [47]:
max_len = int(review_df["length"].max())

In [46]:
review_df  = review_df[review_df["length"]<512]

In [50]:
tokenizer,model = bert_define()

Some weights of the model checkpoint at /remote-home/cs_acmis_wsf/ai4dingo/model/bert_english_pretrained were not used when initializing BertModel: ['cls.seq_relationship.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.decoder.weight', 'cls.seq_relationship.weight', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [51]:
review_df

,review_id,user_id,business_id,stars,useful,funny,cool,text,review_data,length
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3.0,0,0,0,"If you decide to eat here, just be aware it is...",2018-07-07 22:09:11,511
2,saUsX_uimxRlCVr67Z4Jig,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3.0,0,0,0,Family diner. Had the buffet. Eclectic assortm...,2014-02-05 20:30:30,339
3,AqPFMleE6RsU23_auESxiA,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5.0,1,0,1,"Wow! Yummy, different, delicious. Our favo...",2015-01-04 00:01:03,243
5,JrIxlS1TzJ-iCu79ul40cQ,eUta8W_HdHMXPzLBBZhL1A,04UD14gamNjLY0IDYVhHJg,1.0,1,2,1,I am a long term frequent customer of this est...,2015-09-23 23:10:31,341
7,_ZeMknuYdlQcUqng_Im3yg,yfFzsLmaWF2d4Sr0UNbBgg,LHSTtnW3YHCeUkRDGyJOyw,5.0,2,0,0,Amazingly amazing wings and homemade bleu chee...,2015-08-07 02:29:16,192
...,...,...,...,...,...,...,...,...,...,...
6990272,wD5ZWao_vjyT2h4xmGam8Q,7L7GL5Pi2cf8mbm2Dpw4zw,e_E-jq9mwm7wk75k7Yi-Xw,5.0,1,0,1,It is very rare for a restaurant to be this go...,2022-01-17 22:36:01,336
6990273,zHZ-A1qyKDEgyZMDaD--wg,_XVdmFWSgTN6YlojUxixTA,6WaI-IN8ql0xpEKlb4q8tg,5.0,1,0,0,We redesigned my moms dress and mad it complet...,2022-01-17 20:59:01,209
6990275,H0RIamZu0B0Ei0P4aeh3sQ,qskILQ3k0I_qcCMI-k6_QQ,jals67o91gcrD4DC81Vk6w,5.0,1,2,1,Latest addition to services from ICCU is Apple...,2014-12-17 21:45:20,320
6990276,shTPgbgdwTHSuU67mGCmZQ,Zo0th2m8Ez4gLSbHftiQvg,2vLksaMmSEcGbjI5gywpZA,5.0,2,1,2,"This spot offers a great, affordable east week...",2021-03-31 16:55:10,395


In [59]:
vector_count = len(review_df)

In [60]:
vector_count

4237560

In [61]:
n, batch_size, batch = closest_batch_size(vector_count)
print(f"vector_count: {vector_count}, n: {n}, batch_size: {batch_size}, batch: {batch}")

vector_count: 4237560, n: 6, batch_size: 64, batch: 66211


In [62]:
max_vector_count = 2**n * batch

In [63]:
rows_per_part = 4096
n =  math.ceil(vector_count / rows_per_part)

In [64]:
n

1035

In [65]:
split_dfs_review = split_dataframe(review_df, n,rows_per_part)

In [71]:
batch_size_num = 512

for i in range(252,len(split_dfs_review)):
    df = split_dfs_review[i]
    print(f'正在读取第 {i+1} 个DF')
    text = df["text"].tolist()
    
    print(f'初始化batch数据： {i+1}')
    batch_token, batch_segment, batch_mask = list(), list(), list()
    for t in text:
        # text 作 tokenizer 分词
        token = tokenizer.tokenize(t)
        token = ['[CLS]'] + token + ['[SEP]']
        token_id = tokenizer.convert_tokens_to_ids(token)  # 字转换vocab中的index

        # 加padding补齐及segment、mask
        padding = [0] * (max_len - len(token_id))
        mask = [1] * len(token_id) + padding
        segment = [0] * len(token_id) + padding
        token_id = token_id + padding

        batch_token.append(token_id)
        batch_segment.append(segment)
        batch_mask.append(mask)
        
    print(f'batch数据 组装完毕 ：{i+1}')
   
    batch_tensor_token = torch.tensor(batch_token)
    batch_tensor_segment = torch.tensor(batch_segment)
    batch_tensor_mask = torch.tensor(batch_mask)

    batch_len = len(batch_tensor_token)
    
    if torch.cuda.is_available():
        batch_tensor_token = batch_tensor_token.to('cuda:0')
        batch_tensor_segment = batch_tensor_segment.to('cuda:0')
        batch_tensor_mask = batch_tensor_mask.to('cuda:0')
    
    print(f'batch_tensor数据 组装完毕 ：{i+1},开始分块处理')
    
    batch_tensor_token_chunked = [batch_tensor_token[i:i+batch_size_num] for i in range(0,batch_len,batch_size_num)]

    batch_tensor_segment_chunked = [batch_tensor_segment[i:i+batch_size_num] for i in range(0,batch_len,batch_size_num)]

    batch_tensor_mask_chunked = [batch_tensor_mask[i:i+batch_size_num] for i in range(0,batch_len,batch_size_num)]

    print(f'分块处理完毕数据完毕，每块的长度为:{batch_size_num},一共有 {len(batch_tensor_token_chunked)} 块， 开始调用模型')
    
    gc.collect()
    torch.cuda.empty_cache()
    text_vector = []
    
    try:
        for k in range(len(batch_tensor_token_chunked)):
            with torch.no_grad():
                print(f'正在处理 {i+1} ----- {k+1}')
                outputs = model(batch_tensor_token_chunked[k], token_type_ids=batch_tensor_segment_chunked[k], attention_mask=batch_tensor_mask_chunked[k])
                outputs = outputs[0][:, 0, :]  # 取cls向量
                for o in range(outputs.shape[0]):
                    text_vector.append(outputs[o])
        df["feature"] = [ tv.tolist() for tv in  text_vector] 
        df.to_csv(f'/remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/review_{i+1}.csv', index=False)
        print(f"文件写入完毕 {i+1}")
                    
    except Exception as e:
        print(f"出现异常 {i+1}: {e}")
        continue
            
    
    print('============================')
    print()

正在读取第 253 个DF
初始化batch数据： 253
batch数据 组装完毕 ：253
batch_tensor数据 组装完毕 ：253,开始分块处理
分块处理完毕数据完毕，每块的长度为:512,一共有 8 块， 开始调用模型
正在处理 253 ----- 1
正在处理 253 ----- 2
正在处理 253 ----- 3
正在处理 253 ----- 4
正在处理 253 ----- 5
正在处理 253 ----- 6
正在处理 253 ----- 7
正在处理 253 ----- 8


/tmp/ipykernel_474/1690566335.py:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["feature"] = [ tv.tolist() for tv in  text_vector]


文件写入完毕 253

正在读取第 254 个DF
初始化batch数据： 254
batch数据 组装完毕 ：254
batch_tensor数据 组装完毕 ：254,开始分块处理
分块处理完毕数据完毕，每块的长度为:512,一共有 8 块， 开始调用模型
正在处理 254 ----- 1
正在处理 254 ----- 2
正在处理 254 ----- 3
正在处理 254 ----- 4
正在处理 254 ----- 5
正在处理 254 ----- 6
正在处理 254 ----- 7
正在处理 254 ----- 8


/tmp/ipykernel_474/1690566335.py:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["feature"] = [ tv.tolist() for tv in  text_vector]


文件写入完毕 254

正在读取第 255 个DF
初始化batch数据： 255
batch数据 组装完毕 ：255
batch_tensor数据 组装完毕 ：255,开始分块处理
分块处理完毕数据完毕，每块的长度为:512,一共有 8 块， 开始调用模型
正在处理 255 ----- 1
正在处理 255 ----- 2
正在处理 255 ----- 3
正在处理 255 ----- 4
正在处理 255 ----- 5
正在处理 255 ----- 6
正在处理 255 ----- 7
正在处理 255 ----- 8


...

文件写入完毕 1033

正在读取第 1034 个DF
初始化batch数据： 1034
batch数据 组装完毕 ：1034
batch_tensor数据 组装完毕 ：1034,开始分块处理
分块处理完毕数据完毕，每块的长度为:512,一共有 8 块， 开始调用模型
正在处理 1034 ----- 1
正在处理 1034 ----- 2
正在处理 1034 ----- 3
正在处理 1034 ----- 4
正在处理 1034 ----- 5
正在处理 1034 ----- 6
正在处理 1034 ----- 7
正在处理 1034 ----- 8


/tmp/ipykernel_474/1690566335.py:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["feature"] = [ tv.tolist() for tv in  text_vector]


文件写入完毕 1034

正在读取第 1035 个DF
初始化batch数据： 1035
batch数据 组装完毕 ：1035
batch_tensor数据 组装完毕 ：1035,开始分块处理
分块处理完毕数据完毕，每块的长度为:512,一共有 5 块， 开始调用模型
正在处理 1035 ----- 1
正在处理 1035 ----- 2
正在处理 1035 ----- 3
正在处理 1035 ----- 4
正在处理 1035 ----- 5


/tmp/ipykernel_474/1690566335.py:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["feature"] = [ tv.tolist() for tv in  text_vector]


文件写入完毕 1035



### handle tip

In [79]:
tip_data = []

# 打开JSON文件
with open('/remote-home/cs_acmis_wsf/ai4dingo/yelp/yelp_academic_dataset_tip.json', 'r', encoding='utf-8') as file:
    # 逐行读取
    for line in file:
        # 解析每一行为JSON对象
        tip = json.loads(line)
        user_id = tip["user_id"]
        business_id = tip["business_id"]
        text = tip["text"]
        date = tip["date"]
        compliment_count = tip["compliment_count"]
        r = [user_id,business_id,text,date,compliment_count]
        tip_data.append(r)

In [80]:
tip_df = pd.DataFrame(tip_data,columns=["user_id","business_id","text","date","compliment_count"])

In [83]:
tip_df= tip_df.drop_duplicates(subset='text', keep='last')

In [85]:
tip_df["text"] = tip_df["text"].apply(handle_text)

/tmp/ipykernel_474/302868527.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tip_df["text"] = tip_df["text"].apply(handle_text)


In [86]:
tip_df = tip_df.dropna(subset=['text'])

In [88]:
tip_df['length'] = tip_df['text'].str.len()

In [89]:
max_len = int(tip_df["length"].max())

In [91]:
max_len

500

In [92]:
tokenizer,model = bert_define()

Some weights of the model checkpoint at /remote-home/cs_acmis_wsf/ai4dingo/model/bert_english_pretrained were not used when initializing BertModel: ['cls.seq_relationship.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.decoder.weight', 'cls.seq_relationship.weight', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [93]:
vector_count = len(tip_df)

In [94]:
vector_count

851033

In [95]:
n, batch_size, batch = closest_batch_size(vector_count)
print(f"vector_count: {vector_count}, n: {n}, batch_size: {batch_size}, batch: {batch}")

vector_count: 851033, n: 6, batch_size: 64, batch: 13297


In [96]:
rows_per_part = 4096
n =  math.ceil(vector_count / rows_per_part)

In [97]:
n

208

In [98]:
split_dfs_tip = split_dataframe(tip_df, n,rows_per_part)

In [99]:
batch_size_num = 512

for i in range(len(split_dfs_tip)):
    df = split_dfs_tip[i]
    print(f'正在读取第 {i+1} 个DataFrame')
    text = df["text"].tolist()
    
    print(f'初始化batch数据： {i+1}')
    batch_token, batch_segment, batch_mask = list(), list(), list()
    for t in text:
        # text 作 tokenizer 分词
        token = tokenizer.tokenize(t)
        token = ['[CLS]'] + token + ['[SEP]']
        token_id = tokenizer.convert_tokens_to_ids(token)  # 字转换vocab中的index

        # 加padding补齐及segment、mask
        padding = [0] * (max_len - len(token_id))
        mask = [1] * len(token_id) + padding
        segment = [0] * len(token_id) + padding
        token_id = token_id + padding

        batch_token.append(token_id)
        batch_segment.append(segment)
        batch_mask.append(mask)
        
    print(f'batch数据 组装完毕 ：{i+1}')
   
    batch_tensor_token = torch.tensor(batch_token)
    batch_tensor_segment = torch.tensor(batch_segment)
    batch_tensor_mask = torch.tensor(batch_mask)

    batch_len = len(batch_tensor_token)
    
    if torch.cuda.is_available():
        batch_tensor_token = batch_tensor_token.to('cuda:0')
        batch_tensor_segment = batch_tensor_segment.to('cuda:0')
        batch_tensor_mask = batch_tensor_mask.to('cuda:0')
    
    print(f'batch_tensor数据 组装完毕 ：{i+1},开始分块处理')
    
    batch_tensor_token_chunked = [batch_tensor_token[i:i+batch_size_num] for i in range(0,batch_len,batch_size_num)]

    batch_tensor_segment_chunked = [batch_tensor_segment[i:i+batch_size_num] for i in range(0,batch_len,batch_size_num)]

    batch_tensor_mask_chunked = [batch_tensor_mask[i:i+batch_size_num] for i in range(0,batch_len,batch_size_num)]

    print(f'分块处理完毕数据完毕，每块的长度为:{batch_size_num},一共有 {len(batch_tensor_token_chunked)} 块， 开始调用模型')
    
    gc.collect()
    torch.cuda.empty_cache()
    text_vector = []
    
    try:
        for k in range(len(batch_tensor_token_chunked)):
            with torch.no_grad():
                print(f'正在处理 {i+1} ----- {k+1}')
                outputs = model(batch_tensor_token_chunked[k], token_type_ids=batch_tensor_segment_chunked[k], attention_mask=batch_tensor_mask_chunked[k])
                outputs = outputs[0][:, 0, :]  # 取cls向量
                for o in range(outputs.shape[0]):
                    text_vector.append(outputs[o])
        df["feature"] = [ tv.tolist() for tv in  text_vector] 
        df.to_csv(f'/remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/tip_{i+1}.csv', index=False)
        print(f"文件写入完毕 {i+1}")
                    
    except Exception as e:
        print(f"出现异常 {i+1}: {e}")
        continue
            
    
    print('============================')
    print()

正在读取第 1 个DataFrame
初始化batch数据： 1
batch数据 组装完毕 ：1
batch_tensor数据 组装完毕 ：1,开始分块处理
分块处理完毕数据完毕，每块的长度为:512,一共有 8 块， 开始调用模型
正在处理 1 ----- 1
正在处理 1 ----- 2
正在处理 1 ----- 3
正在处理 1 ----- 4
正在处理 1 ----- 5
正在处理 1 ----- 6
正在处理 1 ----- 7
正在处理 1 ----- 8


/tmp/ipykernel_474/2717872624.py:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["feature"] = [ tv.tolist() for tv in  text_vector]


文件写入完毕 1

正在读取第 2 个DataFrame
初始化batch数据： 2
batch数据 组装完毕 ：2
batch_tensor数据 组装完毕 ：2,开始分块处理
分块处理完毕数据完毕，每块的长度为:512,一共有 8 块， 开始调用模型
正在处理 2 ----- 1
正在处理 2 ----- 2
正在处理 2 ----- 3
正在处理 2 ----- 4
正在处理 2 ----- 5
正在处理 2 ----- 6
正在处理 2 ----- 7
正在处理 2 ----- 8


/tmp/ipykernel_474/2717872624.py:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["feature"] = [ tv.tolist() for tv in  text_vector]


文件写入完毕 2

正在读取第 3 个DataFrame
初始化batch数据： 3
batch数据 组装完毕 ：3
batch_tensor数据 组装完毕 ：3,开始分块处理
分块处理完毕数据完毕，每块的长度为:512,一共有 8 块， 开始调用模型
正在处理 3 ----- 1
正在处理 3 ----- 2
正在处理 3 ----- 3
正在处理 3 ----- 4
正在处理 3 ----- 5
正在处理 3 ----- 6
正在处理 3 ----- 7
正在处理 3 ----- 8


...

文件写入完毕 206

正在读取第 207 个DataFrame
初始化batch数据： 207
batch数据 组装完毕 ：207
batch_tensor数据 组装完毕 ：207,开始分块处理
分块处理完毕数据完毕，每块的长度为:512,一共有 8 块， 开始调用模型
正在处理 207 ----- 1
正在处理 207 ----- 2
正在处理 207 ----- 3
正在处理 207 ----- 4
正在处理 207 ----- 5
正在处理 207 ----- 6
正在处理 207 ----- 7
正在处理 207 ----- 8


/tmp/ipykernel_474/2717872624.py:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["feature"] = [ tv.tolist() for tv in  text_vector]


文件写入完毕 207

正在读取第 208 个DataFrame
初始化batch数据： 208
batch数据 组装完毕 ：208
batch_tensor数据 组装完毕 ：208,开始分块处理
分块处理完毕数据完毕，每块的长度为:512,一共有 7 块， 开始调用模型
正在处理 208 ----- 1
正在处理 208 ----- 2
正在处理 208 ----- 3
正在处理 208 ----- 4
正在处理 208 ----- 5
正在处理 208 ----- 6
正在处理 208 ----- 7


/tmp/ipykernel_474/2717872624.py:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["feature"] = [ tv.tolist() for tv in  text_vector]


文件写入完毕 208



In [108]:
tip_df

,user_id,business_id,text,date,compliment_count,length
0,AGNUgVwnZUey3gcPCJ76iw,3uLgwr0qeCNMjKenHJwPGQ,Avengers time with the ladies.,2012-05-18 02:17:21,0,30
1,NBN4MgHP9D3cw--SnauTkA,QoezRbYQncpRqyrLH6Iqjg,They have lots of good deserts and tasty cuban...,2013-02-05 18:35:10,0,57
2,-copOvldyKh1qr-vzkDEvw,MYoRNLb5chwjQe3c_k37Gg,It's open even when you think it isn't,2013-08-18 00:56:08,0,38
3,FjMQVZjSqY8syIO-53KFKw,hV-bABTK-glh5wj31ps_Jw,Very decent fried chicken,2017-06-27 23:05:38,0,25
4,ld0AperBXk1h6UbqmM80zw,_uN0OudeJ3Zl_tf6nxg5ww,Appetizers.. platter special for lunch,2012-10-06 19:43:09,0,38
...,...,...,...,...,...,...
908910,eYodOTF8pkqKPzHkcxZs-Q,3lHTewuKFt5IImbXJoFeDQ,Disappointed in one of your managers.,2021-09-11 19:18:57,0,37
908911,1uxtQAuJ2T5Xwa_wp7kUnA,OaGf0Dp56ARhQwIDT90w_g,Great food and service.,2021-10-30 11:54:36,0,23
908912,v48Spe6WEpqehsF2xQADpg,hYnMeAO77RGyTtIzUSKYzQ,Love their Cubans!!,2021-11-05 13:18:56,0,19
908913,ckqKGM2hl7I9Chp5IpAhkw,s2eyoTuJrcP7I_XyjdhUHQ,Great pizza great price,2021-11-20 16:11:44,0,23
